<a href="https://colab.research.google.com/github/Arfa-Tariq/learning-ai-engineering/blob/main/projects/04-building-systems/04-Chain_of_Thought_Reasoning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Process Inputs: Chain of Thought Reasoning

In [1]:
!pip install -q groq

from groq import Groq
from google.colab import userdata
import tiktoken

GROQ_API_KEY = userdata.get('groq_key')
client = Groq(api_key=GROQ_API_KEY)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 2.9 MB/s eta 0:00:00


In [3]:
def get_completion_from_messages(messages,
                                 model="llama-3.3-70b-versatile",
                                 temperature=0,):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, # this is the degree of randomness of the model's output
    )
    return response.choices[0].message.content

## Chain-of-Thought Prompting

In [4]:
delimiter = "####"
system_message = f"""
Follow these steps to answer the customer queries.
The customer query will be delimited with four hashtags,\
i.e. {delimiter}.

Step 1:{delimiter} First decide whether the user is \
asking a question about a specific product or products. \
Product cateogry doesn't count.

Step 2:{delimiter} If the user is asking about \
specific products, identify whether \
the products are in the following list.
All available products:
1. Product: TechPro Ultrabook
   Category: Computers and Laptops
   Brand: TechPro
   Model Number: TP-UB100
   Warranty: 1 year
   Rating: 4.5
   Features: 13.3-inch display, 8GB RAM, 256GB SSD, Intel Core i5 processor
   Description: A sleek and lightweight ultrabook for everyday use.
   Price: $799.99

2. Product: BlueWave Gaming Laptop
   Category: Computers and Laptops
   Brand: BlueWave
   Model Number: BW-GL200
   Warranty: 2 years
   Rating: 4.7
   Features: 15.6-inch display, 16GB RAM, 512GB SSD, NVIDIA GeForce RTX 3060
   Description: A high-performance gaming laptop for an immersive experience.
   Price: $1199.99

3. Product: PowerLite Convertible
   Category: Computers and Laptops
   Brand: PowerLite
   Model Number: PL-CV300
   Warranty: 1 year
   Rating: 4.3
   Features: 14-inch touchscreen, 8GB RAM, 256GB SSD, 360-degree hinge
   Description: A versatile convertible laptop with a responsive touchscreen.
   Price: $699.99

4. Product: TechPro Desktop
   Category: Computers and Laptops
   Brand: TechPro
   Model Number: TP-DT500
   Warranty: 1 year
   Rating: 4.4
   Features: Intel Core i7 processor, 16GB RAM, 1TB HDD, NVIDIA GeForce GTX 1660
   Description: A powerful desktop computer for work and play.
   Price: $999.99

5. Product: BlueWave Chromebook
   Category: Computers and Laptops
   Brand: BlueWave
   Model Number: BW-CB100
   Warranty: 1 year
   Rating: 4.1
   Features: 11.6-inch display, 4GB RAM, 32GB eMMC, Chrome OS
   Description: A compact and affordable Chromebook for everyday tasks.
   Price: $249.99

Step 3:{delimiter} If the message contains products \
in the list above, list any assumptions that the \
user is making in their \
message e.g. that Laptop X is bigger than \
Laptop Y, or that Laptop Z has a 2 year warranty.

Step 4:{delimiter}: If the user made any assumptions, \
figure out whether the assumption is true based on your \
product information.

Step 5:{delimiter}: First, politely correct the \
customer's incorrect assumptions if applicable. \
Only mention or reference products in the list of \
5 available products, as these are the only 5 \
products that the store sells. \
Answer the customer in a friendly tone.

Use the following format:
Step 1:{delimiter} <step 1 reasoning>
Step 2:{delimiter} <step 2 reasoning>
Step 3:{delimiter} <step 3 reasoning>
Step 4:{delimiter} <step 4 reasoning>
Response to user:{delimiter} <response to customer>

Make sure to include {delimiter} to separate every step.
"""

In [5]:
user_message = f"""
by how much is the BlueWave Chromebook more expensive \
than the TechPro Desktop"""

messages =  [
{'role':'system',
 'content': system_message},
{'role':'user',
 'content': f"{delimiter}{user_message}{delimiter}"},
]

response = get_completion_from_messages(messages)
print(response)

Step 1:#### The user is asking a question about the price difference between two specific products, the BlueWave Chromebook and the TechPro Desktop, which suggests they are inquiring about a comparison between these products.

Step 2:#### The products mentioned, BlueWave Chromebook and TechPro Desktop, are both in the list of available products. The BlueWave Chromebook has a price of $249.99, and the TechPro Desktop has a price of $999.99.

Step 3:#### The user is assuming that the BlueWave Chromebook is more expensive than the TechPro Desktop. This assumption can be identified in the question as it asks by how much the BlueWave Chromebook is more expensive than the TechPro Desktop.

Step 4:#### The assumption that the BlueWave Chromebook is more expensive than the TechPro Desktop is incorrect. According to the product information, the TechPro Desktop ($999.99) is actually more expensive than the BlueWave Chromebook ($249.99).

Response to user:#### I think there might be a misundersta

In [6]:
user_message = f"""
do you sell tvs"""
messages =  [
{'role':'system',
 'content': system_message},
{'role':'user',
 'content': f"{delimiter}{user_message}{delimiter}"},
]
response = get_completion_from_messages(messages)
print(response)

Step 1:#### The user is asking a general question about the products sold, but not specifically about a product or products.

Step 2:#### The user is not asking about any specific products from the list, but rather inquiring about a product category (TVs) that is not present in the list.

Step 3:#### The user is assuming that the store might sell TVs, which is not explicitly stated in the available product information.

Step 4:#### Based on the provided product list, the store only sells computers and laptops, with no mention of TVs.

Response to user:#### Hi there, we actually specialize in computers and laptops. We have a range of options from ultrabooks to gaming laptops and desktops. Unfortunately, we don't carry TVs in our store. If you're looking for a new computer or laptop, I'd be happy to help you find the perfect one!


## Inner Monologue
- Since we asked the LLM to separate its reasoning steps by a delimiter, we can hide the chain-of-thought reasoning from the final output that the user sees.

In [7]:
try:
    final_response = response.split(delimiter)[-1].strip()
except Exception as e:
    final_response = "Sorry, I'm having trouble right now, please try asking another question."

print(final_response)

Hi there, we actually specialize in computers and laptops. We have a range of options from ultrabooks to gaming laptops and desktops. Unfortunately, we don't carry TVs in our store. If you're looking for a new computer or laptop, I'd be happy to help you find the perfect one!
